In [ ]:
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch


## 1. Setup

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving v2.zip to v2.zip


In [ ]:
import zipfile
import os

zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall('/content/project')

print("Extracted successfully!")
print(os.listdir('/content/project'))

Extracted successfully!
['demo']


Make sure `processed/` is empty before preprocessing.

In [ ]:
%cd /content/project/demo

!rm -rf data/processed
!mkdir -p data/processed

/content/project/demo


## 2. Preprocessing
Runs the original `prepare_data.py`.

In [ ]:
!python prepare_data.py

data/processed/train_clean.csv: 14000 rows  {'neutral': 5264, 'negative': 4102, 'positive': 4634}
data/processed/dev_clean.csv: 3000 rows  {'positive': 982, 'negative': 890, 'neutral': 1128}
data/processed/test_clean.csv: 3000 rows  {'neutral': 1100, 'negative': 900, 'positive': 1000}


## 3. Data checks

In [ ]:
import pandas as pd

files = {
    "train": "data/processed/train_clean.csv",
    "dev": "data/processed/dev_clean.csv",
    "test": "data/processed/test_clean.csv"
}

for name, path in files.items():
    df = pd.read_csv(path)

    print(f"\n===== {name.upper()} =====")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print("\nSentiment distribution:")
    print(df["sentiment"].value_counts())
    print("\nMissing values:")
    print(df.isna().sum())


===== TRAIN =====
Rows: 14000
Columns: ['uid', 'text', 'clean_text', 'sentiment', 'lang_tags', 'hin_ratio']

Sentiment distribution:
sentiment
neutral     5264
positive    4634
negative    4102
Name: count, dtype: int64

Missing values:
uid           0
text          0
clean_text    1
sentiment     0
lang_tags     0
hin_ratio     0
dtype: int64

===== DEV =====
Rows: 3000
Columns: ['uid', 'text', 'clean_text', 'sentiment', 'lang_tags', 'hin_ratio']

Sentiment distribution:
sentiment
neutral     1128
positive     982
negative     890
Name: count, dtype: int64

Missing values:
uid           0
text          0
clean_text    0
sentiment     0
lang_tags     0
hin_ratio     0
dtype: int64

===== TEST =====
Rows: 3000
Columns: ['uid', 'text', 'clean_text', 'sentiment', 'lang_tags', 'hin_ratio']

Sentiment distribution:
sentiment
neutral     1100
positive    1000
negative     900
Name: count, dtype: int64

Missing values:
uid           0
text          0
clean_text    0
sentiment     0
lang_tags

Number of empty `clean_text` rows after preprocessing (1 in train, none in dev/test).

In [ ]:
import pandas as pd

files = {
    "train": "data/processed/train_clean.csv",
    "dev": "data/processed/dev_clean.csv",
    "test": "data/processed/test_clean.csv"
}

for name, path in files.items():
    df = pd.read_csv(path)

    empty = df["clean_text"].isna() | (df["clean_text"].fillna("").str.strip() == "")

    print(f"\n===== {name.upper()} =====")
    print("Empty clean_text:", empty.sum())
    print("Percentage:", round(empty.mean() * 100, 3), "%")

    if empty.sum():
        print("\nBy sentiment:")
        print(df.loc[empty, "sentiment"].value_counts())


===== TRAIN =====
Empty clean_text: 1
Percentage: 0.007 %

By sentiment:
sentiment
neutral    1
Name: count, dtype: int64

===== DEV =====
Empty clean_text: 0
Percentage: 0.0 %

===== TEST =====
Empty clean_text: 0
Percentage: 0.0 %


## 4. Label conflicts and final training set
Identical `clean_text` with different labels is noise. Most such groups are exact ties, so they cannot be fixed by majority vote and are removed from train.

In [ ]:
import pandas as pd

files = {
    "train": "data/processed/train_clean.csv",
    "dev": "data/processed/dev_clean.csv",
    "test": "data/processed/test_clean.csv"
}

for name, path in files.items():
    df = pd.read_csv(path)

    # Ignore the single empty clean_text case
    df = df[df["clean_text"].notna()].copy()

    # Find clean_text values appearing more than once
    counts = df["clean_text"].value_counts()
    duplicated_clean = counts[counts > 1]

    # Keep all rows involved in collisions
    dup = df[df["clean_text"].isin(duplicated_clean.index)].copy()

    # Number of clean_text groups with multiple labels
    conflicting = (
        dup.groupby("clean_text")["sentiment"]
        .nunique()
    )

    conflicting = conflicting[conflicting > 1]

    print(f"\n===== {name.upper()} =====")
    print("Rows involved in clean_text collisions:", len(dup))
    print("Unique duplicated clean_text:", len(duplicated_clean))
    print("Conflicting-label groups:", len(conflicting))
    print("Rows in conflicting groups:",
          dup[dup["clean_text"].isin(conflicting.index)].shape[0])


===== TRAIN =====
Rows involved in clean_text collisions: 309
Unique duplicated clean_text: 123
Conflicting-label groups: 47
Rows in conflicting groups: 147

===== DEV =====
Rows involved in clean_text collisions: 28
Unique duplicated clean_text: 10
Conflicting-label groups: 3
Rows in conflicting groups: 6

===== TEST =====
Rows involved in clean_text collisions: 17
Unique duplicated clean_text: 5
Conflicting-label groups: 1
Rows in conflicting groups: 2


In [ ]:
# Train: find conflicting clean_text groups (same cleaned text, different labels)
TRAIN = Path("/content/project/demo/data/processed/train_clean.csv")
df = pd.read_csv(TRAIN)

print("Train rows:", len(df))

groups = (
    df.groupby("clean_text")["sentiment"]
    .agg(total_rows="size", unique_labels="nunique")
    .reset_index()
)
conflicts = groups[groups["unique_labels"] > 1]

majority_clear = 0
exact_tie = 0

for text in conflicts["clean_text"]:
    counts = df.loc[df["clean_text"] == text, "sentiment"].value_counts()
    n_max = (counts == counts.max()).sum()
    if n_max == 1:
        majority_clear += 1
    else:
        exact_tie += 1

print("Conflicting groups:", len(conflicts))
print("Groups with clear majority:", majority_clear)
print("Groups with exact tie:", exact_tie)


In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# PATHS
# ============================================================

TRAIN = Path("/content/project/demo/data/processed/train_clean.csv")
OUT = Path("/content/project/demo/data/processed/train_final.csv")


# ============================================================
# LOAD
# ============================================================

df = pd.read_csv(TRAIN)

print("Original training rows:", len(df))


# ============================================================
# FIND CONFLICTING CLEAN_TEXT GROUPS
# ============================================================

label_counts = (
    df.groupby("clean_text")["sentiment"]
    .nunique()
)

conflicting_texts = label_counts[
    label_counts > 1
].index


# ============================================================
# REMOVE CONFLICTING GROUPS
# ============================================================

train_final = df[
    ~df["clean_text"].isin(conflicting_texts)
].copy()


# ============================================================
# SAVE
# ============================================================

train_final.to_csv(
    OUT,
    index=False,
    encoding="utf-8"
)


# ============================================================
# REPORT
# ============================================================

print("Conflicting groups removed:", len(conflicting_texts))
print("Rows removed:", len(df) - len(train_final))
print("Final training rows:", len(train_final))

print("\nFinal label distribution:")
print(train_final["sentiment"].value_counts())

print("\nSaved to:")
print(OUT)

Original training rows: 14000
Conflicting groups removed: 47
Rows removed: 147
Final training rows: 13853

Final label distribution:
sentiment
neutral     5180
positive    4598
negative    4075
Name: count, dtype: int64

Saved to:
/content/project/demo/data/processed/train_final.csv


In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path("/content/project/demo/data/processed")

train = pd.read_csv(BASE / "train_final.csv")
dev = pd.read_csv(BASE / "dev_clean.csv")
test = pd.read_csv(BASE / "test_clean.csv")

print("=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

print(f"\nTrain: {len(train)} rows")
print(f"Dev:   {len(dev)} rows")
print(f"Test:  {len(test)} rows")

print("\nLabel distribution")
print("\nTRAIN:")
print(train["sentiment"].value_counts())

print("\nDEV:")
print(dev["sentiment"].value_counts())

print("\nTEST:")
print(test["sentiment"].value_counts())

print("\nColumns:")
print(train.columns.tolist())

print("\nSample training text:")
print(train[["clean_text", "sentiment"]].head(3).to_string(index=False))

DATASET SUMMARY

Train: 13853 rows
Dev:   3000 rows
Test:  3000 rows

Label distribution

TRAIN:
sentiment
neutral     5180
positive    4598
negative    4075
Name: count, dtype: int64

DEV:
sentiment
neutral     1128
positive     982
negative     890
Name: count, dtype: int64

TEST:
sentiment
neutral     1100
positive    1000
negative     900
Name: count, dtype: int64

Columns:
['uid', 'text', 'clean_text', 'sentiment', 'lang_tags', 'hin_ratio']

Sample training text:
                                                                                                       clean_text sentiment
nen á vist bolest vztek smutek zmatek osam ě lost beznad ě j a nakonec jen klid asi takhle vypad á m ů j life ...   neutral
        haan yaar neha 😔😔 kab karega woh post 😭 usne na sach mein photoshoot karna chahiye phir woh post karega …   neutral
       television media congress ke liye nhi h . ye toh aapko pata chal hi gya hoga . achha hoga ki congress ke …  negative


## 5. MuRIL fine-tuning
Trained on `train_final.csv`, model selected on dev macro-F1, saved to `models/muril_final`.

In [ ]:
# ============================================================
# MuRIL Fine-tuning
# ============================================================

import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

# ------------------------------------------------------------
# 1. Load datasets
# ------------------------------------------------------------

train_df = pd.read_csv("/content/project/demo/data/processed/train_final.csv")
dev_df   = pd.read_csv("/content/project/demo/data/processed/dev_clean.csv")

print("Train:", train_df.shape)
print("Dev:  ", dev_df.shape)

# ------------------------------------------------------------
# 2. Label mapping
# ------------------------------------------------------------

label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

train_df["label"] = train_df["sentiment"].map(label2id)
dev_df["label"]   = dev_df["sentiment"].map(label2id)

# Use clean_text as the model input
train_ds = Dataset.from_pandas(
    train_df[["clean_text", "label"]],
    preserve_index=False
)

dev_ds = Dataset.from_pandas(
    dev_df[["clean_text", "label"]],
    preserve_index=False
)

# ------------------------------------------------------------
# 3. Load MuRIL
# ------------------------------------------------------------

MODEL_NAME = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    texts = [str(x) for x in batch["clean_text"]]

    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
dev_ds = dev_ds.map(tokenize, batched=True)

train_ds = train_ds.remove_columns(["clean_text"])
dev_ds   = dev_ds.remove_columns(["clean_text"])

train_ds.set_format("torch")
dev_ds.set_format("torch")

# ------------------------------------------------------------
# 4. Load model
# ------------------------------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

# ------------------------------------------------------------
# 5. Evaluation metrics
# ------------------------------------------------------------

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1
    }

# ------------------------------------------------------------
# 6. Training configuration
# ------------------------------------------------------------

training_args = TrainingArguments(
    output_dir="/content/project/demo/models/muril",

    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    logging_steps=100,
    report_to="none",

    fp16=torch.cuda.is_available()
)

# ------------------------------------------------------------
# 7. Trainer
# ------------------------------------------------------------

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

# ------------------------------------------------------------
# 8. Train
# ------------------------------------------------------------

trainer.train()

# ------------------------------------------------------------
# 9. Final DEV evaluation
# ------------------------------------------------------------

results = trainer.evaluate()

print("\n" + "=" * 70)
print("MuRIL DEV RESULTS")
print("=" * 70)

for key, value in results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

# ------------------------------------------------------------
# 10. Save model
# ------------------------------------------------------------

SAVE_PATH = "/content/project/demo/models/muril_final"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("\nSaved MuRIL model to:")
print(SAVE_PATH)

Train: (13853, 6)
Dev:   (3000, 6)


Map:   0%|          | 0/13853 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.624748,0.842763,4,0.652000,0.656084



MuRIL DEV RESULTS
eval_loss: 0.8428
eval_accuracy: 0.6520
eval_macro_f1: 0.6561


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved MuRIL model to:
/content/project/demo/models/muril_final


## 6. Final evaluation on the test set
The test set is used only here, once.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# ============================================================
# LOAD TEST DATA
# ============================================================

TEST_PATH = "/content/project/demo/data/processed/test_clean.csv"
MODEL_PATH = "/content/project/demo/models/muril_final"

test_df = pd.read_csv(TEST_PATH)

print("Test rows:", len(test_df))
print("Columns:", test_df.columns.tolist())

# ============================================================
# LABEL MAPPING
# ============================================================

label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

test_df["label"] = test_df["sentiment"].map(label2id)

# ============================================================
# DATASET
# ============================================================

test_ds = Dataset.from_pandas(
    test_df[["clean_text", "label"]],
    preserve_index=False
)

# ============================================================
# TOKENIZER + MODEL
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

# ============================================================
# TOKENIZE
# ============================================================

def tokenize(batch):
    return tokenizer(
        batch["clean_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

test_tokenized = test_ds.map(
    tokenize,
    batched=True
)

test_tokenized = test_tokenized.remove_columns(
    ["clean_text"]
)

test_tokenized.set_format("torch")

# ============================================================
# PREDICTIONS
# ============================================================

from torch.utils.data import DataLoader
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
model.eval()

loader = DataLoader(
    test_tokenized,
    batch_size=16,
    shuffle=False
)

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in loader:

        labels = batch.pop("label")

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        outputs = model(**batch)

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        ).cpu().numpy()

        all_predictions.extend(predictions)
        all_labels.extend(labels.numpy())

# ============================================================
# RESULTS
# ============================================================

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro"
)

print()
print("=" * 70)
print("MURIL TEST RESULTS")
print("=" * 70)

print(f"Accuracy : {accuracy:.4f}")
print(f"Macro F1 : {macro_f1:.4f}")

print()
print("Classification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "negative",
            "neutral",
            "positive"
        ],
        digits=4
    )
)

print()
print("Confusion Matrix:")

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print(
    pd.DataFrame(
        cm,
        index=[
            "Actual Negative",
            "Actual Neutral",
            "Actual Positive"
        ],
        columns=[
            "Pred Negative",
            "Pred Neutral",
            "Pred Positive"
        ]
    )
)

Test rows: 3000
Columns: ['uid', 'text', 'clean_text', 'sentiment', 'lang_tags', 'hin_ratio']


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]


MURIL TEST RESULTS
Accuracy : 0.7147
Macro F1 : 0.7195

Classification Report:
              precision    recall  f1-score   support

    negative     0.7497    0.7256    0.7374       900
     neutral     0.6210    0.6718    0.6454      1100
    positive     0.8009    0.7520    0.7757      1000

    accuracy                         0.7147      3000
   macro avg     0.7239    0.7165    0.7195      3000
weighted avg     0.7196    0.7147    0.7164      3000


Confusion Matrix:
                 Pred Negative  Pred Neutral  Pred Positive
Actual Negative            653           228             19
Actual Neutral             193           739            168
Actual Positive             25           223            752


## 7. Inference demo

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_PATH = "/content/project/demo/models/muril_final"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

model.eval()

# Check label mapping
print("Label mapping:", model.config.id2label)


def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=-1)
    predicted_id = torch.argmax(probabilities, dim=-1).item()

    label = model.config.id2label[predicted_id]
    confidence = probabilities[0][predicted_id].item()

    return label, confidence


# Test examples
test_texts = [
    "Modi ji ne bahut achha kaam kiya hai",
    "Ye government bilkul bekaar hai",
    "Aaj mausam thik hai",
    "I am very happy today",
    "This is the worst decision ever"
]

for text in test_texts:
    label, confidence = predict_sentiment(text)

    print("\nText:", text)
    print("Prediction:", label)
    print("Confidence:", f"{confidence:.2%}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Label mapping: {0: 'negative', 1: 'neutral', 2: 'positive'}

Text: Modi ji ne bahut achha kaam kiya hai
Prediction: positive
Confidence: 85.83%

Text: Ye government bilkul bekaar hai
Prediction: negative
Confidence: 65.56%

Text: Aaj mausam thik hai
Prediction: neutral
Confidence: 48.36%

Text: I am very happy today
Prediction: positive
Confidence: 83.63%

Text: This is the worst decision ever
Prediction: neutral
Confidence: 69.94%
